### 1. Importing and cleaning the dataset

In [ ]:
import pandas as pd
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

TODAY = pd.Timestamp("2026-06-11") # pinned for reproducibility

INPUT_PATH  = "../data/processed/filtered_bb_sme_sectors.zip"
OUTPUT_PATH = "../data/processed/company_profiles_scored.csv"
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

In [2]:
df = pd.read_csv(INPUT_PATH, low_memory=False)
df.columns = df.columns.str.strip()

DATE_COLS = [
    "IncorporationDate",
    "Accounts.NextDueDate",
    "Accounts.LastMadeUpDate",
    "ConfStmtNextDueDate",
    "ConfStmtLastMadeUpDate",
] + [f"PreviousName_{i}.CONDATE" for i in range(1, 11)]

for col in DATE_COLS:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")

print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df[["CompanyName", "sector", "segment", "Accounts.AccountCategory"]].head(3)

Loaded 869,043 rows, 57 columns


,CompanyName,sector,segment,Accounts.AccountCategory
0,!NFLECTION ADVISORY LIMITED,"Technology, legal & professional",BB/SME,TOTAL EXEMPTION FULL
1,!NFOGENIE LTD,"Technology, legal & professional",BB,MICRO ENTITY
2,!NNOV8 LIMITED,Fast growth & emerging,BB,MICRO ENTITY


In [11]:
df.dtypes

CompanyName                                      str
CompanyNumber                                    str
RegAddress.CareOf                                str
RegAddress.POBox                                 str
RegAddress.AddressLine1                          str
RegAddress.AddressLine2                          str
RegAddress.PostTown                              str
RegAddress.County                                str
RegAddress.Country                               str
RegAddress.PostCode                              str
CompanyCategory                                  str
CompanyStatus                                    str
CountryOfOrigin                                  str
DissolutionDate                              float64
IncorporationDate                     datetime64[us]
Accounts.AccountRefDay                       float64
Accounts.AccountRefMonth                     float64
Accounts.NextDueDate                  datetime64[us]
Accounts.LastMadeUpDate               datetime

In [13]:
df['RegAddress.CareOf'].value_counts()

RegAddress.CareOf
HILLIER HOPKINS LLP      59
TAXASSIST ACCOUNTANTS    54
PETS AT HOME LIMITED     30
CONNECT ACCOUNTING       28
C/O                      25
                         ..
MR R SCAWN                1
MARK LODGE                1
ZINIR LTD                 1
MR M NIA                  1
C/O GERLINDE GNIEWOSZ     1
Name: count, Length: 3403, dtype: int64

### 2. Financial Health Metrics

For now I will limit myself to static metrics derived from the compliance and mortgage register columns. Later on we could link up to the Companies house API and retrieve P&L balance sheets for a more detailed breakdown.

I create the following metrics:
1. Compamny age in years
2. Debt ratio: Measure the share of mortage charges still outstanding; equal to zero for companies with no charge history.
3. 

In [3]:
# M1: Company age in years
df["company_age_years"] = (TODAY - df["IncorporationDate"]).dt.days / 365.25

print("company_age_years:")
print(df["company_age_years"].describe().round(2))


company_age_years:
count    869043.00
mean         11.48
std           9.94
min           0.22
25%           4.65
50%           8.80
75%          15.03
max         169.70
Name: company_age_years, dtype: float64


In [4]:
# M2: Debt ratio
df["debt_ratio"] = (
    df["Mortgages.NumMortOutstanding"]
    / df["Mortgages.NumMortCharges"].clip(lower=1)
)
df.loc[df["Mortgages.NumMortCharges"] == 0, "debt_ratio"] = 0.0

print("debt_ratio (0 = all clear or no history, 1 = all charges still outstanding):")
print(df["debt_ratio"].describe().round(3))
print(f"\nCompanies with >=1 charge: {(df['Mortgages.NumMortCharges'] >= 1).sum():,}")
print(f"Mean debt_ratio for those: {df.loc[df['Mortgages.NumMortCharges'] >= 1, 'debt_ratio'].mean():.3f}")

debt_ratio (0 = all clear or no history, 1 = all charges still outstanding):
count    869043.000
mean          0.063
std           0.231
min           0.000
25%           0.000
50%           0.000
75%           0.000
max           1.000
Name: debt_ratio, dtype: float64

Companies with >=1 charge: 90,778
Mean debt_ratio for those: 0.601
